# Resolve Wikipedia references

Resolve English Wikipedia pages for configured V3 organism groups that have at least one completed positive Phase 4 pathogen assessment. Lookups use accepted species names, follow Wikipedia normalization and redirects, and only save an association when the requested title resolves to an article. Progress is checkpointed after every API batch, so interrupted runs resume from the CSV.

By default, the notebook uses `run.organism_groups` from `configs/default.yaml` (currently `viruses`). It limits each selected group to species with at least one completed `candidate_pathogen_evidence=True` record in the Phase 4 pathogen decisions table. Change the group setting to `bacteria`, `fungi`, or any combination to resolve those pathogen candidates. Completed records are reused; set `FORCE_REFRESH = True` to repeat lookups for the selected groups.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pathogen_atlas_v3").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Open this notebook from the Pathogen Atlas version3 repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.common.config import selected_organism_groups
CONFIG = load_yaml(PROJECT_ROOT / "configs/default.yaml")
ORGANISM_GROUPS = selected_organism_groups(CONFIG)
SPECIES_PATH = PROJECT_ROOT / CONFIG["paths"]["taxonomy_release"] / "species.parquet"
PATHOGEN_DECISIONS_PATH = PROJECT_ROOT / CONFIG["paths"]["release_root"] / "pathogen_decisions.parquet"
OUTPUT_PATH = PROJECT_ROOT / "data/06_references/release/species_wikipedia_links.csv"
FORCE_REFRESH = False

for required_path in (SPECIES_PATH, PATHOGEN_DECISIONS_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Run Phases 1 and 4 first; required input is missing: {required_path}")

species = pd.read_parquet(SPECIES_PATH, columns=["organism_id", "organism_group", "accepted_name", "rank"])
decisions = pd.read_parquet(PATHOGEN_DECISIONS_PATH, columns=["organism_id", "organism_group", "candidate_pathogen_evidence", "status"])
pathogen_ids = set(decisions.loc[
    decisions.organism_group.isin(ORGANISM_GROUPS)
    & decisions.status.astype(str).eq("completed")
    & decisions.candidate_pathogen_evidence.fillna(False).astype(bool),
    "organism_id",
].astype(str))
scope = species.loc[
    species.organism_group.isin(ORGANISM_GROUPS)
    & species["rank"].astype(str).str.casefold().eq("species")
    & species.organism_id.astype(str).isin(pathogen_ids)
]
existing_count = 0
if OUTPUT_PATH.exists():
    previous = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
    existing_count = int(previous.organism_group.isin(ORGANISM_GROUPS).sum()) if "organism_group" in previous else 0

display(Markdown(
    f"**Organism groups:** {', '.join(ORGANISM_GROUPS)}  ·  "
    f"**Species-rank records:** {scope.organism_id.nunique():,}  ·  "
    f"**Existing associations in scope:** {existing_count:,}  ·  "
    f"**Force refresh:** {FORCE_REFRESH}"
))


**Organism groups:** fungi  ·  **Species-rank records:** 440  ·  **Existing associations in scope:** 0  ·  **Force refresh:** False

## Resolve and save Wikipedia associations

The resolver writes a table keyed by `organism_id`, so identical accepted names in different domains remain separate. The same output file can be extended incrementally as you switch between bacteria, fungi, and viruses. Only the selected groups’ pathogen candidates are included.

In [2]:
from pathogen_atlas_v3.references.links import resolve_wikipedia_links

links = resolve_wikipedia_links(
    SPECIES_PATH,
    OUTPUT_PATH,
    ORGANISM_GROUPS,
    force_refresh=FORCE_REFRESH,
    organism_ids=pathogen_ids,
)
selected_links = links.loc[links.organism_group.isin(ORGANISM_GROUPS)]
found = selected_links.match_status.eq("found")
display(Markdown(
    f"**Saved:** `{OUTPUT_PATH.relative_to(PROJECT_ROOT)}`  ·  "
    f"**Records:** {len(selected_links):,}  ·  "
    f"**Wikipedia articles found:** {int(found.sum()):,}  ·  "
    f"**No exact page:** {int((~found).sum()):,}"
))
display(selected_links.head(12))


Resolving 440 records; reusing 257 completed rows.


Resolved 25/440 pending records.


Resolved 50/440 pending records.


Resolved 75/440 pending records.


Resolved 100/440 pending records.


Resolved 125/440 pending records.


... (12 outputs left out)

Resolved 440/440 pending records.


Saved 697 associations to /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version3/data/06_references/release/species_wikipedia_links.csv


**Saved:** `data/06_references/release/species_wikipedia_links.csv`  ·  **Records:** 440  ·  **Wikipedia articles found:** 121  ·  **No exact page:** 319

,organism_id,organism_group,accepted_name,wikipedia_title,wikipedia_url,match_status,match_type
0,fungi:0007843d65dafa45c0d6,fungi,Nakaseomyces nivariensis,,,not_found,exact_title_query
1,fungi:006bae437be3781c7102,fungi,Neocucurbitaria unguis-hominis,,,not_found,exact_title_query
2,fungi:009ec7bab8bb984f4bcb,fungi,Inonotus tropicalis,,,not_found,exact_title_query
3,fungi:011a0c22a6c1dc0cc8a9,fungi,Wickerhamiella pararugosa,,,not_found,exact_title_query
4,fungi:011b86f54ee97e2c0ab3,fungi,Naganishia albida,Naganishia albida,https://en.wikipedia.org/wiki/Naganishia_albida,found,exact_or_redirect
5,fungi:018727e860669fa4fc72,fungi,Scedosporium boydii,,,not_found,exact_title_query
6,fungi:0300cd9102a7e01b6272,fungi,Simplicillium sinense,,,not_found,exact_title_query
7,fungi:0367452d738926eba164,fungi,Blastobotrys allociferrii,,,not_found,exact_title_query
8,fungi:03701fa31aa5f45c7e99,fungi,Madurella mycetomatis,Madurella mycetomatis,https://en.wikipedia.org/wiki/Madurella_myceto...,found,exact_or_redirect
9,fungi:04a19f50889fe2b5ef8b,fungi,Microsporum ferrugineum,,,not_found,exact_title_query
